<a href="https://colab.research.google.com/github/krjeyy/AI-ML-lab-experiments/blob/main/k_means_and_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-Means Clustering and Interpretation
This notebook demonstrates how to load a dataset, preprocess it, determine the optimal number of clusters using the Elbow Method, run the K-Means algorithm, and interpret the final clusters.

### 📁 Loading Your Own CSV File
To use your own dataset, upload your CSV file to Colab's local storage (using the file icon on the left panel) and run the cell below to load it into a pandas DataFrame.

In [ ]:
import urllib.request

# Download the standard driver dataset from a public repository
url = 'https://raw.githubusercontent.com/madhug-nadig/Machine-Learning-Algorithms-from-Scratch/master/data/driver-data.csv'
csv_path = 'driver-data.csv'

try:
    urllib.request.urlretrieve(url, csv_path)
    print("✅ Success: 'driver-data.csv' has been downloaded and is ready to use!")
except Exception as e:
    print(f"❌ Failed to download dataset: {e}")

❌ Failed to download dataset: HTTP Error 404: Not Found


In [ ]:
import pandas as pd
# 1. Replace 'your_file.csv' with the path to your uploaded CSV file
csv_path = 'driver-data.csv'

# 2. Load the dataset
df = pd.read_csv(csv_path)

# 3. Preview your dataset
display(df.head())

# 4. Print columns so you can see which features you want to cluster
print("Available columns:", list(df.columns))

FileNotFoundError: [Errno 2] No such file or directory: 'driver-data.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Create a synthetic customer dataset for demonstration
# In practice, you would load your CSV using: df = pd.read_csv('your_file.csv')
np.random.seed(42)
data = {
    'CustomerID': range(1, 201),
    'Age': np.random.randint(18, 70, 200),
    'Annual_Income_k': np.random.randint(15, 137, 200),
    'Spending_Score': np.random.randint(1, 101, 200)
}
df = pd.DataFrame(data)
display(df.head())

In [ ]:
import numpy as np

# Activation functions and their derivatives
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Input dataset (4 samples, 3 features)
X = np.array([
    [0, 0, 1],
    [0, 1, 1],
    [1, 0, 1],
    [1, 1, 1]
])

# Target output (4 samples, 1 output)
y = np.array([[0], [1], [1], [0]])

# Seed random numbers for reproducibility
np.random.seed(42)

# Initialize weights randomly with mean 0
input_dim = 3
hidden_dim = 4
output_dim = 1

weights_0 = 2 * np.random.random((input_dim, hidden_dim)) - 1
weights_1 = 2 * np.random.random((hidden_dim, output_dim)) - 1

learning_rate = 0.1
epochs = 20000

print("Training the Neural Network...")
for epoch in range(epochs):
    # --- 1. Forward Propagation ---
    # Layer 0 (Input) to Layer 1 (Hidden)
    layer_0 = X
    layer_1 = sigmoid(np.dot(layer_0, weights_0))

    # Layer 1 (Hidden) to Layer 2 (Output)
    layer_2 = sigmoid(np.dot(layer_1, weights_1))

    # --- 2. Backward Propagation ---
    # Calculate error at the output layer
    layer_2_error = y - layer_2

    if (epoch + 1) % 5000 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss (MSE): {np.mean(np.square(layer_2_error)):.6f}")

    # Calculate delta for the output layer (Gradient)
    layer_2_delta = layer_2_error * sigmoid_derivative(layer_2)

    # Backpropagate error to the hidden layer
    layer_1_error = layer_2_delta.dot(weights_1.T)

    # Calculate delta for the hidden layer
    layer_1_delta = layer_1_error * sigmoid_derivative(layer_1)

    # --- 3. Update Weights (Gradient Descent) ---
    weights_1 += layer_1.T.dot(layer_2_delta) * learning_rate
    weights_0 += layer_0.T.dot(layer_1_delta) * learning_rate

print("\nFinal predictions after training:")
display(layer_2)

Training the Neural Network...
Epoch 5000/20000 - Loss (MSE): 0.009347
Epoch 10000/20000 - Loss (MSE): 0.002133
Epoch 15000/20000 - Loss (MSE): 0.001157
Epoch 20000/20000 - Loss (MSE): 0.000786

Final predictions after training:


array([[0.01832683],
       [0.96917832],
       [0.97279207],
       [0.03344309]])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Preprocessing: Select features and scale them
# K-Means is highly sensitive to the scale of the data, so we standardize it first
features = ['Annual_Income_k', 'Spending_Score']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Use the Elbow Method to find the optimal number of clusters (K)
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method to Determine Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.grid(True)
plt.show()

In [ ]:
# Apply K-Means with the chosen number of clusters (e.g., K=5 is common for this pattern)
k_optimal = 5
kmeans = KMeans(n_clusters=k_optimal, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize the clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='Annual_Income_k',
    y='Spending_Score',
    hue='Cluster',
    palette='Set1',
    data=df,
    s=100,
    alpha=0.8
)
plt.title('Customer Clusters based on Income and Spending Score')
plt.xlabel('Annual Income ($k)')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

In [ ]:
# Interpret the clusters by looking at their average feature values
cluster_summary = df.groupby('Cluster')[features].mean().reset_index()
display(cluster_summary)